# Embedding and RAG with Milvus and OpenAI

## Embedding Quick example

In [ ]:
import os
from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings

load_dotenv()

embeddings = OpenAIEmbeddings(
    model=os.getenv("AI_EMBEDDING_MODEL"),
    base_url=os.getenv("AI_ENDPOINT"),
    api_key=os.getenv("AI_API_KEY")
)

test_embedding = embeddings.embed_query("This is a test")
print(len(test_embedding))
print(test_embedding[:10])

## Embedding Docs for RAG demo

In [ ]:
from glob import glob
from tqdm import tqdm

text_lines = []
for file_path in glob("milvus_docs/en/faq/*.md", recursive=True):
    with open(file_path, "r") as file:
        file_text = file.read()
    text_lines += file_text.split("# ")

data = []
for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": embeddings.embed_query(line), "text": line})

## Creating a new empty Milvus Collection

In [ ]:
from pymilvus import MilvusClient
milvus_client = MilvusClient(uri="./milvus_demo.db")
collection_name = "rag_collection"

if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=len(data[0]['vector']),
    metric_type="IP",  # Inner product distance
    consistency_level="Strong",  # Strong consistency level
)

## Inserting text embedded previously to Milvus collection

In [ ]:
milvus_client.insert(collection_name=collection_name, data=data)

## Queryng the collection

In [ ]:
question = "How is data stored in milvus?"

search_res = milvus_client.search(
    collection_name=collection_name,
    data=[
        embeddings.embed_query(question)
        ],
    limit=3,  # Return top 3 results
    search_params={"metric_type": "IP", "params": {}},  # Inner product distance
    output_fields=["text"],  # Return the text field
)

In [ ]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

## Joining the best 3 answers to build context paragraph

In [ ]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

## Creating Instance of GenIA model to answer to the question based on Milvus top 3 results

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

model = ChatOpenAI(
    model=os.getenv("AI_MODEL"),
    base_url=os.getenv("AI_ENDPOINT"),
    api_key=os.getenv("AI_API_KEY")
)

SYSTEM_PROMPT = """
Human: You are an AI assistant. You are able to find answers to the questions from the contextual passage snippets provided.
"""
USER_PROMPT = f"""
Use the following pieces of information enclosed in <context> tags to provide an answer to the question enclosed in <question> tags.
<context>
{context}
</context>
<question>
{question}
</question>
"""

messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=USER_PROMPT),
    ]

print("👤 User: \n" + question)
print("\n📋 Context from Milvus Search result: \n" + context)
print("\n🤖 AI: \n", end="", flush=True)
full_response = ""
for chunk in model.stream(messages):
    if chunk.content:
        print(chunk.content, end="", flush=True)
        full_response += chunk.content